# Extraction directinfo.ma — Renseignements juridiques par N° RC

Ce notebook (Google Colab) traite une **liste de N° RC** (saisie ou fichier Excel) :
1. ouvre la **Recherche avancée** de www.directinfo.ma dans un navigateur Chromium sans interface (Playwright) ;
2. saisit chaque **Numéro RC** (et, en option, la ville) en gardant la même session ;
3. **affiche le code de sécurité** : vous le tapez vous-même (une seule fois si le site ne le redemande pas) ;
4. parcourt **toutes les pages de résultats**, ouvre chaque fiche `fiche-detail/...` ;
5. extrait le tableau **RENSEIGNEMENTS JURIDIQUES** (Dénomination, Statut juridique, Tribunal, N° de RC, Activité, Forme juridique, Date d'immatriculation, Capital…) ;
6. ajoute les résultats à un **fichier Excel cumulatif** sur votre Google Drive (sans doublons) et en télécharge une copie.

> Respectez les conditions d'utilisation de directinfo.ma / OMPIC. Le script garde une pause entre chaque fiche.

**Mode d'emploi** : `Exécution > Tout exécuter`, puis répondez aux questions (liste des RC en cellule 3, codes en cellule 4).

In [ ]:
# 1) Installation (≈ 1 min, une seule fois par session Colab)
!pip -q install playwright beautifulsoup4 openpyxl pandas
!playwright install --with-deps chromium > /dev/null 2>&1 && echo "Chromium OK"

In [ ]:
# 2) Paramètres et fonctions
import asyncio, re, time, unicodedata
from urllib.parse import urljoin
import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display, Image
from playwright.async_api import async_playwright

BASE_URL       = "https://www.directinfo.ma"
SEARCH_URL     = BASE_URL + "/recherche-multi-criteres"
PAUSE_SEC      = 1.5      # pause entre deux fiches (politesse envers le site)

# Fichier Excel cumulatif : chaque lancement y ajoute les nouvelles sociétés.
USE_DRIVE         = True   # True = fichier conservé sur votre Google Drive ; False = fichier local Colab
FICHIER_CUMUL     = "/content/drive/MyDrive/directinfo/directinfo_base.xlsx"
TELECHARGER_COPIE = True   # télécharger aussi une copie sur le PC à la fin
MAX_PAGES      = 50       # garde-fou pagination
HEADLESS       = True

def norm(s: str) -> str:
    """minuscule, sans accents ni espaces superflus (pour comparer des libellés)."""
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip().lower()

def clean(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

async def field_after_label(page, label_text, tag="input"):
    """Trouve le champ (input/select) qui suit un libellé visible, ex. 'Numéro RC'."""
    # a) <label for=...> correctement associé
    try:
        loc = page.get_by_label(label_text, exact=False)
        if await loc.count() > 0:
            return loc.first
    except Exception:
        pass
    # b) premier champ qui suit le texte du libellé dans le DOM
    xp = (f"xpath=//*[self::label or self::span or self::p or self::div]"
          f"[contains(normalize-space(.), '{label_text}') and not(.//{tag})]"
          f"/following::{tag}[not(@type='hidden')][1]")
    loc = page.locator(xp)
    if await loc.count() > 0:
        return loc.first
    raise RuntimeError(f"Champ introuvable pour le libellé « {label_text} »")

async def captcha_image(page):
    """Localise l'image du CAPTCHA."""
    for sel in ["img[src*='captcha' i]", "img[id*='captcha' i]", "img[class*='captcha' i]",
                "xpath=//*[contains(normalize-space(.),'Saisir le code')]/following::img[1]"]:
        loc = page.locator(sel)
        if await loc.count() > 0:
            return loc.first
    return None

def fiche_links(html):
    soup = BeautifulSoup(html, "html.parser")
    out = []
    for a in soup.select("a[href*='fiche-detail']"):
        url = urljoin(BASE_URL, a["href"])
        if url not in out:
            out.append(url)
    return out

def parse_renseignements(html, url):
    """Extrait le tableau 'RENSEIGNEMENTS JURIDIQUES' d'une fiche."""
    soup = BeautifulSoup(html, "html.parser")
    row = {"URL fiche": url}
    title = None
    for h in soup.find_all(["h1", "h2", "h3", "h4", "div", "span"]):
        if norm(h.get_text()) == "renseignements juridiques":
            title = h
            break
    table = title.find_next("table") if title else soup.find("table")
    if table is None:
        row["Erreur"] = "Tableau Renseignements juridiques introuvable"
        return row
    for tr in table.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        if len(cells) >= 2:
            key = clean(cells[0].get_text(" "))
            val = clean(cells[1].get_text(" "))
            if key:
                row[key] = val
    # Documents disponibles (Etats de synthèse, Statuts, …) affichés à droite du tableau
    docs = [clean(x.get_text(" ")) for x in soup.find_all(["a", "button"])
            if re.search(r"\(\d+\)\s*$", clean(x.get_text(" ")))]
    if docs:
        row["Documents disponibles"] = " | ".join(dict.fromkeys(docs))
    years = sorted(set(re.findall(r"Etats de synth[eè]se\s+(\d{4})", soup.get_text(" "))))
    if years:
        row["Années états de synthèse"] = ", ".join(years)
    return row

async def next_page(page):
    """Clique sur 'page suivante' si elle existe. Renvoie True si la page a changé."""
    candidates = ["a[rel='next']", "li.next:not(.disabled) a", ".pagination li.active + li a",
                  "a:has-text('Suivant')", "a:has-text('»')", "a:has-text('›')"]
    for sel in candidates:
        loc = page.locator(sel)
        if await loc.count() > 0 and await loc.first.is_visible():
            before = page.url + str(len(await page.content()))
            await loc.first.click()
            await page.wait_for_load_state("networkidle")
            after = page.url + str(len(await page.content()))
            return before != after
    return False

async def submit_search(page, rc, ville, demander_code=True, diag=None):
    """Charge le formulaire, le remplit et l'envoie. Renvoie True si une page de résultats est obtenue.
    Le code de sécurité est affiché et saisi APRÈS le chargement, sur cette même page :
    recharger la page générerait un nouveau code et rendrait la saisie invalide."""
    await page.goto(SEARCH_URL, wait_until="networkidle")
    await (await field_after_label(page, "Numéro RC")).fill(str(rc))
    if ville:
        v = await field_after_label(page, "Ville", tag="select")
        try:
            await v.select_option(label=ville.upper())
        except Exception:
            print(f"    Ville « {ville} » absente de la liste : ignorée.")
    champ_code = await field_after_label(page, "Saisir le code")
    if demander_code:
        code = await ask_code(page)
        await champ_code.fill(code)

    # Bouton « Rechercher » DU FORMULAIRE AVANCÉ (et non celui de la barre de recherche du bandeau).
    form = champ_code.locator("xpath=ancestor::form[1]")
    btn = form.locator("button:has-text('Rechercher'), input[type='submit'][value*='echercher'], "
                       "button[type='submit'], input[type='submit']")
    if await form.count() > 0 and await btn.count() > 0:
        await btn.first.click()
    else:                                   # pas de <form> englobant : on valide avec Entrée
        await champ_code.press("Enter")
    try:
        await page.wait_for_load_state("networkidle", timeout=20000)
    except Exception:
        pass
    await page.wait_for_timeout(800)

    html = await page.content()
    texte = norm(await page.inner_text("body"))
    ok = bool(fiche_links(html)) or "nombre de resultats" in texte
    if not ok and diag:
        # Diagnostic : URL obtenue, messages d'erreur visibles, capture et HTML de la page.
        print("    URL après envoi :", page.url)
        msgs = await page.locator("[class*='error' i], [class*='alert' i], [class*='invalid' i], "
                                  "[class*='danger' i], .help-block").all_inner_texts()
        msgs = [clean(m) for m in msgs if clean(m)]
        print("    Message(s) du site :", " | ".join(msgs[:5]) if msgs else "(aucun message d'erreur repéré)")
        await page.screenshot(path=f"{diag}.png", full_page=True)
        open(f"{diag}.html", "w", encoding="utf-8").write(html)
        print(f"    Capture + page enregistrées : {diag}.png / {diag}.html (panneau Fichiers à gauche)")
    return ok

async def ask_code(page):
    img = await captcha_image(page)
    if img is None:
        print("    (aucune image de code trouvée sur la page)")
        return ""
    display(Image(data=await img.screenshot(), width=420))   # affiché en grand
    return input("    Tapez le code affiché ci-dessus : ").strip()

async def search_one(page, rc, ville, state):
    """Recherche un RC (en réutilisant la session) et renvoie la liste des fiches extraites."""
    ok = False
    # Le site redemande-t-il le code à chaque recherche ? On le vérifie une seule fois,
    # sur la 2e recherche, en envoyant le formulaire sans code.
    if state["code_par_recherche"] is None and state["nb_recherches"] > 0:
        ok = await submit_search(page, rc, ville, demander_code=False)
        state["code_par_recherche"] = not ok
        print("    → le site " + ("redemande" if not ok else "ne redemande pas") + " le code à chaque recherche")
    elif state["code_par_recherche"] is False:
        ok = await submit_search(page, rc, ville, demander_code=False)
    for attempt in range(1, 4):
        if ok:
            break
        ok = await submit_search(page, rc, ville, demander_code=True,
                                 diag=f"refus_RC{rc}_tentative{attempt}")
        if not ok:
            print(f"    Recherche refusée (code incorrect ?) — tentative {attempt}/3")
    if not ok:
        raise RuntimeError("échec de la recherche après 3 tentatives")
    state["nb_recherches"] += 1

    m = re.search(r"Nombre de R[ée]sultats trouv[ée]s\s*:\s*(?:<[^>]+>)*\s*(\d+)", await page.content())
    print("    Résultats annoncés :", m.group(1) if m else "?")

    links = []
    for n in range(MAX_PAGES):
        new = [u for u in fiche_links(await page.content()) if u not in links]
        links += new
        print(f"    page {n+1} : {len(new)} fiche(s)  (total {len(links)})")
        if not new or not await next_page(page):
            break

    rows = []
    for i, url in enumerate(links, 1):
        if url in state["deja_vues"]:                       # fiche déjà extraite pour un autre RC
            rows.append(dict(state["deja_vues"][url]))
            continue
        try:
            await page.goto(url, wait_until="networkidle")
            r = parse_renseignements(await page.content(), url)
        except Exception as e:
            r = {"URL fiche": url, "Erreur": str(e)}
        state["deja_vues"][url] = r
        rows.append(dict(r))
        print(f"    [{i}/{len(links)}] {r.get('Dénomination', r.get('Erreur', ''))}")
        await asyncio.sleep(PAUSE_SEC)
    return rows

async def search_many(demandes, exact=True):
    """demandes : liste de (rc, ville_ou_None). Renvoie (DataFrame fiches, DataFrame journal)."""
    toutes, journal = [], []
    state = {"code_par_recherche": None, "nb_recherches": 0, "deja_vues": {}}
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        ctx = await browser.new_context(locale="fr-FR", viewport={"width": 1400, "height": 1000})
        page = await ctx.new_page()
        for k, (rc, ville) in enumerate(demandes, 1):
            print(f"\n[{k}/{len(demandes)}] RC {rc}" + (f" — {ville}" if ville else ""))
            try:
                rows = await search_one(page, rc, ville, state)
                gardees = [r for r in rows
                           if not exact or str(r.get("N° de RC", "")).strip() == str(rc)]
                for r in gardees:
                    r["RC recherché"], r["Ville recherchée"] = rc, ville or ""
                toutes += gardees
                journal.append({"RC": rc, "Ville": ville or "", "Statut": "OK",
                                "Fiches trouvées": len(rows), "Fiches retenues": len(gardees)})
            except Exception as e:
                print("    ERREUR :", e)
                journal.append({"RC": rc, "Ville": ville or "", "Statut": f"Erreur : {e}",
                                "Fiches trouvées": 0, "Fiches retenues": 0})
        await browser.close()

    df = pd.DataFrame(toutes)
    ordre = ["RC recherché", "Ville recherchée", "Dénomination", "Statut juridique", "Tribunal",
             "N° de RC", "Activité", "Forme juridique", "Date d'immatriculation", "Capital (en MAD)",
             "Documents disponibles", "Années états de synthèse", "URL fiche", "Erreur"]
    if not df.empty:
        df = df[[c for c in ordre if c in df.columns] + [c for c in df.columns if c not in ordre]]
    return df, pd.DataFrame(journal)

In [ ]:
# 3) Liste des RC à rechercher
#    - soit tapez-les séparés par des virgules / points-virgules (ex. : 2319, 4512;TIFELT, 87211;CASABLANCA)
#      « RC;VILLE » permet de préciser la ville pour un RC donné ;
#    - soit laissez vide pour importer un fichier Excel/CSV avec une colonne « RC »
#      (et, en option, une colonne « Ville »).
from google.colab import files

saisie = input("N° RC (séparés par des virgules), ou vide pour importer un fichier : ").strip()
demandes = []
if saisie:
    for item in re.split(r"[,\n]+", saisie):
        if item.strip():
            rc, _, ville = item.partition(";")
            demandes.append((rc.strip(), ville.strip() or None))
else:
    up = files.upload()
    nom_fichier = next(iter(up))
    src = pd.read_csv(nom_fichier, dtype=str, sep=None, engine="python") if nom_fichier.lower().endswith(".csv") \
          else pd.read_excel(nom_fichier, dtype=str)
    cols = {norm(c): c for c in src.columns}
    c_rc = next((cols[c] for c in cols if c in ("rc", "n rc", "numero rc", "n° de rc", "registre de commerce")), src.columns[0])
    c_ville = next((cols[c] for c in cols if c in ("ville", "tribunal")), None)
    for _, r in src.iterrows():
        rc = str(r[c_rc]).strip()
        if rc and rc.lower() != "nan":
            rc = rc[:-2] if rc.endswith(".0") else rc
            v = str(r[c_ville]).strip() if c_ville and str(r[c_ville]).lower() != "nan" else None
            demandes.append((rc, v or None))

demandes = list(dict.fromkeys(demandes))     # sans doublons, ordre conservé
EXACT = input("Ne garder que les fiches dont le N° de RC est exactement égal ? (o/n) [o] : ").strip().lower() != "n"
print(f"{len(demandes)} recherche(s) à lancer :", demandes[:10], "..." if len(demandes) > 10 else "")

In [ ]:
# 4) Lancement (tapez les codes au fur et à mesure qu'ils s'affichent)
df, journal = await search_many(demandes, exact=EXACT)
print(f"\n{len(df)} société(s) extraite(s) pour {len(demandes)} RC")
display(journal)
display(df)

In [ ]:
# 5) Ajout au fichier Excel cumulatif sur Google Drive (+ téléchargement d'une copie)
#    Chaque lancement complète le même fichier : pas de doublon (même fiche = ligne mise à jour).
from openpyxl.utils import get_column_letter
import os

if USE_DRIVE:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")            # autorisation demandée une seule fois par session
    chemin = FICHIER_CUMUL
    os.makedirs(os.path.dirname(chemin), exist_ok=True)
else:
    chemin = os.path.basename(FICHIER_CUMUL)

def ajouter(chemin, feuille, nouveau, cle=None):
    # Concatène 'nouveau' aux lignes déjà présentes dans la feuille, en dédoublonnant sur 'cle'.
    if os.path.exists(chemin):
        try:
            ancien = pd.read_excel(chemin, sheet_name=feuille, dtype=str)
        except ValueError:
            ancien = pd.DataFrame()
    else:
        ancien = pd.DataFrame()
    tout = pd.concat([ancien, nouveau.astype(str)], ignore_index=True)
    if cle and cle in tout.columns:
        tout = tout.drop_duplicates(subset=[cle], keep="last")
    return tout.replace({"nan": "", "None": ""})

horodatage = time.strftime("%d/%m/%Y %H:%M")
nouv = df.copy()
nouv.insert(0, "Date extraction", horodatage)
jn = journal.copy()
jn.insert(0, "Date", horodatage)
feuilles = {
    "Renseignements juridiques": ajouter(chemin, "Renseignements juridiques", nouv, cle="URL fiche"),
    "Journal": ajouter(chemin, "Journal", jn),
}
with pd.ExcelWriter(chemin, engine="openpyxl") as xw:
    for feuille, data in feuilles.items():
        data.to_excel(xw, index=False, sheet_name=feuille)
        ws = xw.sheets[feuille]
        ws.freeze_panes = "A2"
        if not data.empty:
            ws.auto_filter.ref = ws.dimensions
        for i, col in enumerate(data.columns, 1):
            largeur = max([len(str(col))] + [len(str(v)) for v in data[col].fillna("")])
            ws.column_dimensions[get_column_letter(i)].width = min(largeur + 2, 60)

total = len(feuilles["Renseignements juridiques"])
print(f"{len(df)} société(s) ajoutée(s) / mise(s) à jour — {total} au total dans : {chemin}")
if TELECHARGER_COPIE:
    files.download(chemin)

### Notes
- **Fichier cumulatif** : `MyDrive/directinfo/directinfo_base.xlsx`. Une fiche déjà présente (même URL) est mise à jour, pas dupliquée ; la colonne « Date extraction » indique quand. Ouvrez-le directement depuis Google Drive ou le PC (Drive pour ordinateur). Pour repartir de zéro, supprimez ou renommez le fichier.
- **Codes de sécurité** : le script vérifie, dès la 2e recherche, si le site redemande le code. Si non, vous ne le tapez qu'une fois pour tout le lot ; si oui, il s'affiche à chaque RC.
- **Erreur sur un RC** : le lot continue ; l'erreur est notée dans la feuille « Journal ».
- **« Champ introuvable »** : le site a changé de mise en page — ajoutez `await page.screenshot(path="debug.png")` dans `submit_search` pour voir la page.
- **Une seule page de résultats lue alors que le site en annonce plus** : adaptez la liste `candidates` dans `next_page` au libellé du bouton de pagination réel.